# Figure 1B (bottom) & Figure 1F: Cohort Behavioral Comparisons

| Field | Value |
|---|---|
| **Manuscript panels / stats** | Fig 1B bottom (log RT Welch t217=2.91, p=0.008; % correct Welch t217=-1.05, n.s.), Fig 1F (recognition d' Welch t210=-6.14, p<0.001) |
| **Inputs** | `learn_df_RWH.csv`, `full_df_RWH.csv` (EMU); `learn_df_prolific_v2.csv`, `full_data_df_prolific_v2.csv` (online) |
| **Outputs** | `results/hierarchical_pyem/cohort_comparison/{cohort_logrt_density.pdf,cohort_accuracy_density.pdf,cohort_dprime_density.pdf,cohort_comparison_stats.csv}` |
| **Dependencies** | `membandit`, `numpy`, `pandas`, `scipy`, `matplotlib` |
| **Expected runtime** | < 10 seconds |
| **Upstream steps** | `00_prolific_preprocessing.ipynb` |


# Cohort behavior comparison — EMU vs Prolific (online)

Lightweight descriptive comparison (no RL modeling):

1. **Learning-task log RT** — subject-mean `log(gamble_rt)` from `learn_df`
2. **Subject-level recognition d′** — Day-1 subject `dprime` from combined recognition tables (not image memorability)
3. **Learning-task % correct** — subject mean accuracy on gambling trials (optimal: safe in odd blocks, gamble in even blocks)

Overlapping density curves with Welch two-sample *t*-test annotations.

Outputs → `results/hierarchical_pyem/cohort_comparison/`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
from scipy import stats

SHOW_PLOTS = True
PUB_DPI = 300
ALPHA = 0.05

from membandit import paths
LOAD_EMU = Path(paths.get_emu_behavior_dir())
LOAD_PROLIFIC = Path(paths.get_prolific_dir())
OUT_DIR = Path(paths.results_root()) / "hierarchical_pyem" / "cohort_comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EMU_COLOR = "black"
ONLINE_COLOR = "#8a8a8a"

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": PUB_DPI,
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print({"OUT_DIR": str(OUT_DIR), "LOAD_EMU": str(LOAD_EMU), "LOAD_PROLIFIC": str(LOAD_PROLIFIC)})

## Load and summarize subject-level measures

- **log RT:** mean of `log(gamble_rt)` over learning trials with `gamble_rt > 0` and finite.
- **d′:** Day-1 subject-level recognition `dprime` (one value per participant).
- **accuracy:** subject mean of trial-wise correctness on all learning trials; correct when `gamble=0` in odd blocks and `gamble=1` in even blocks (stored as 0–1 proportion).

In [ ]:
DAY1_LABELS = {"day1", "day_1", "day 1"}


def _is_day1(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower().isin(DAY1_LABELS)


def subject_mean_log_rt(learn_path: Path, cohort: str) -> pd.DataFrame:
    df = pd.read_csv(learn_path)
    if "good_dm_performance" in df.columns:
        df = df.loc[df["good_dm_performance"].astype(bool)].copy()
    rt = pd.to_numeric(df["gamble_rt"], errors="coerce")
    ok = np.isfinite(rt) & (rt > 0)
    df = df.loc[ok].copy()
    df["log_rt"] = np.log(rt.loc[ok].to_numpy(dtype=float))
    out = (
        df.groupby("participant", sort=False)["log_rt"]
        .mean()
        .rename("log_rt")
        .reset_index()
    )
    out["cohort"] = cohort
    out["cohort_label"] = "EMU" if cohort == "EMU" else "online"
    return out


def subject_day1_dprime(combined_path: Path, cohort: str) -> pd.DataFrame:
    df = pd.read_csv(combined_path)
    if "good_dm_performance" in df.columns:
        df = df.loc[df["good_dm_performance"].astype(bool)].copy()
    day1 = df.loc[_is_day1(df["condition"])].copy()
    if len(day1) == 0:
        raise ValueError(f"{combined_path.name}: no Day1 rows for dprime")
    day1["dprime"] = pd.to_numeric(day1["dprime"], errors="coerce")
    out = (
        day1.groupby("participant", sort=False)["dprime"]
        .first()
        .dropna()
        .rename("dprime")
        .reset_index()
    )
    out["cohort"] = cohort
    out["cohort_label"] = "EMU" if cohort == "EMU" else "online"
    return out


def subject_learning_accuracy(learn_path: Path, cohort: str) -> pd.DataFrame:
    """Subject mean accuracy: safe (0) in odd blocks, gamble (1) in even blocks."""
    df = pd.read_csv(learn_path)
    if "good_dm_performance" in df.columns:
        df = df.loc[df["good_dm_performance"].astype(bool)].copy()
    block = pd.to_numeric(df["block"], errors="coerce")
    gamble = pd.to_numeric(df["gamble"], errors="coerce")
    ok = np.isfinite(block) & np.isfinite(gamble)
    df = df.loc[ok].copy()
    optimal = (block.loc[ok] % 2 == 0).to_numpy(dtype=float)  # even->1, odd->0
    df["correct"] = gamble.loc[ok].to_numpy(dtype=float) == optimal
    out = (
        df.groupby("participant", sort=False)["correct"]
        .mean()
        .rename("accuracy")
        .reset_index()
    )
    out["cohort"] = cohort
    out["cohort_label"] = "EMU" if cohort == "EMU" else "online"
    return out


log_rt = pd.concat(
    [
        subject_mean_log_rt(LOAD_EMU / "learn_df_RWH.csv", "EMU"),
        subject_mean_log_rt(LOAD_PROLIFIC / "learn_df_prolific_v2.csv", "Prolific"),
    ],
    ignore_index=True,
)

dprime = pd.concat(
    [
        subject_day1_dprime(LOAD_EMU / "full_df_RWH.csv", "EMU"),
        subject_day1_dprime(LOAD_PROLIFIC / "full_data_df_prolific_v2.csv", "Prolific"),
    ],
    ignore_index=True,
)

accuracy = pd.concat(
    [
        subject_learning_accuracy(LOAD_EMU / "learn_df_RWH.csv", "EMU"),
        subject_learning_accuracy(LOAD_PROLIFIC / "learn_df_prolific_v2.csv", "Prolific"),
    ],
    ignore_index=True,
)

subject_summary = (
    log_rt.merge(
        dprime[["participant", "cohort", "dprime"]],
        on=["participant", "cohort"],
        how="outer",
    ).merge(
        accuracy[["participant", "cohort", "accuracy"]],
        on=["participant", "cohort"],
        how="outer",
    )
)
subject_summary["cohort_label"] = np.where(
    subject_summary["cohort"] == "EMU", "EMU", "online"
)

summary_path = OUT_DIR / "cohort_subject_summary.csv"
subject_summary.to_csv(summary_path, index=False)
print("wrote", summary_path)

for measure, col in [("log_rt", "log_rt"), ("dprime", "dprime"), ("accuracy", "accuracy")]:
    print(f"\n=== {measure} ===")
    rows = []
    for cohort, g in subject_summary.groupby("cohort", sort=False):
        vals = pd.to_numeric(g[col], errors="coerce").dropna()
        rows.append(
            {
                "measure": measure,
                "cohort": cohort,
                "n": int(len(vals)),
                "mean": float(vals.mean()),
                "sd": float(vals.std(ddof=1)),
                "median": float(vals.median()),
            }
        )
    display(pd.DataFrame(rows).round(4))

display(subject_summary.head())


## Density plots with Welch *t*-test

Kernel density estimates for EMU (black) vs online (grey). Annotation is `n.s.` if p >= 0.05, otherwise the exact *p* value.

In [ ]:
def welch_compare(a: np.ndarray, b: np.ndarray) -> dict:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    t, p = stats.ttest_ind(a, b, equal_var=False, nan_policy="omit")
    return {
        "n_emu": int(len(a)),
        "n_online": int(len(b)),
        "mean_emu": float(np.mean(a)),
        "mean_online": float(np.mean(b)),
        "sd_emu": float(np.std(a, ddof=1)),
        "sd_online": float(np.std(b, ddof=1)),
        "t": float(t),
        "p": float(p),
        "sig_label": "n.s." if p >= ALPHA else (f"p = {p:.3g}"),
    }


def plot_cohort_density(
    values_by_label: dict[str, np.ndarray],
    *,
    xlabel: str,
    out_path: Path,
    title: str | None = None,
    ax=None,
    xlim: tuple[float, float] | None = None,
    ylim: tuple[float, float] = (0.0, 2.0),
    yticks: list[float] | None = None,
    x_as_percent: bool = False,
) -> dict:
    emu = np.asarray(values_by_label["EMU"], dtype=float)
    online = np.asarray(values_by_label["online"], dtype=float)
    emu = emu[np.isfinite(emu)]
    online = online[np.isfinite(online)]
    cmp = welch_compare(emu, online)

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(4.2, 3.4))
    else:
        fig = ax.figure

    # Gaussian KDE with shared evaluation grid
    if xlim is not None:
        lo, hi = float(xlim[0]), float(xlim[1])
    else:
        lo = float(min(emu.min(), online.min()))
        hi = float(max(emu.max(), online.max()))
        pad = 0.08 * (hi - lo) if hi > lo else 0.1
        lo, hi = lo - pad, hi + pad
    xs = np.linspace(lo, hi, 256)

    y_hi = float(ylim[1])
    label_cap = y_hi * 0.875
    for label, vals, color in [
        ("EMU", emu, EMU_COLOR),
        ("online", online, ONLINE_COLOR),
    ]:
        if len(vals) < 2 or np.nanstd(vals) == 0:
            continue
        kde = stats.gaussian_kde(vals)
        dens = kde(xs)
        ax.plot(xs, dens, color=color, lw=2.0, label=label)
        i_mode = int(np.argmax(dens))
        ax.text(
            xs[i_mode],
            min(float(dens[i_mode] * 1.02), label_cap),
            label,
            color=color,
            ha="center",
            va="bottom",
            fontsize=11,
            fontweight="medium",
        )

    ax.set_ylim(ylim)
    ax.set_yticks([1, 2] if yticks is None else yticks)
    if xlim is not None:
        ax.set_xlim(xlim)
    if x_as_percent:
        xticks = [0.4, 1.0]
        if xlim is not None:
            xticks = [t for t in xticks if xlim[0] <= t <= xlim[1]]
            if not xticks:
                xticks = [xlim[0], xlim[1]]
        ax.set_xticks(xticks)
        ax.set_xticklabels([f"{int(round(100 * t))}" for t in xticks])

    # Significance bar between cohort means (near top of y-scale)
    bar_y = y_hi * 0.925
    tick = (y_hi - float(ylim[0])) * 0.03
    x0, x1 = cmp["mean_emu"], cmp["mean_online"]
    if x0 > x1:
        x0, x1 = x1, x0
    if xlim is not None:
        x0 = float(np.clip(x0, xlim[0], xlim[1]))
        x1 = float(np.clip(x1, xlim[0], xlim[1]))
    ax.plot([x0, x1], [bar_y, bar_y], color="black", lw=1.2, solid_capstyle="butt")
    ax.plot([x0, x0], [bar_y - tick, bar_y], color="black", lw=1.2)
    ax.plot([x1, x1], [bar_y - tick, bar_y], color="black", lw=1.2)
    ax.text(
        0.5 * (x0 + x1),
        bar_y + tick * 0.35,
        cmp["sig_label"],
        ha="center",
        va="bottom",
        fontsize=11,
        color="black",
    )

    ax.set_xlabel(xlabel)
    ax.set_ylabel("density")
    if title:
        ax.set_title(title, loc="left")

    if own_fig:
        fig.tight_layout()
        fig.savefig(out_path, dpi=PUB_DPI, bbox_inches="tight")
        if SHOW_PLOTS:
            plt.show()
        else:
            plt.close(fig)
        print("wrote", out_path)

    return cmp


stats_rows = []

# --- log RT ---
logrt_by = {
    "EMU": subject_summary.loc[subject_summary["cohort"] == "EMU", "log_rt"].to_numpy(),
    "online": subject_summary.loc[subject_summary["cohort"] == "Prolific", "log_rt"].to_numpy(),
}
cmp_logrt = plot_cohort_density(
    logrt_by,
    xlabel="log RT (s)",
    out_path=OUT_DIR / "cohort_logrt_density.pdf",
    xlim=(-2, 2),
)
stats_rows.append({"measure": "log_rt", **cmp_logrt})

# --- Day1 dprime (exclude d' < 0, both cohorts) ---
# Recognition memory cohort inclusion: EMU d' > 0.25 (n=19); online d' >= 0 (n=193)
dprime_val = pd.to_numeric(subject_summary["dprime"], errors="coerce")
is_emu = subject_summary["cohort"] == "EMU"
dprime_ok = subject_summary.loc[(is_emu & (dprime_val > 0.25)) | (~is_emu & (dprime_val >= 0))].copy()
assert int((dprime_ok["cohort"] == "EMU").sum()) == 19, "Expected n=19 EMU participants"
assert int((dprime_ok["cohort"] == "Prolific").sum()) == 193, "Expected n=193 online participants (df=210)"
dprime_by = {
    "EMU": dprime_ok.loc[dprime_ok["cohort"] == "EMU", "dprime"].to_numpy(),
    "online": dprime_ok.loc[dprime_ok["cohort"] == "Prolific", "dprime"].to_numpy(),
}
n_excl = (
    subject_summary["dprime"].notna().sum() - dprime_ok["dprime"].notna().sum()
)
print(f"d' density: excluded {int(n_excl)} subjects with d' < 0")
cmp_dprime = plot_cohort_density(
    dprime_by,
    xlabel="d'",
    out_path=OUT_DIR / "cohort_dprime_density.pdf",
)
stats_rows.append({"measure": "dprime", **cmp_dprime})

# --- Learning accuracy (% correct) ---
accuracy_by = {
    "EMU": subject_summary.loc[subject_summary["cohort"] == "EMU", "accuracy"].to_numpy(),
    "online": subject_summary.loc[subject_summary["cohort"] == "Prolific", "accuracy"].to_numpy(),
}
cmp_accuracy = plot_cohort_density(
    accuracy_by,
    xlabel="% correct",
    out_path=OUT_DIR / "cohort_accuracy_density.pdf",
    xlim=(0.35, 1.05),
    ylim=(0.0, 5.0),
    yticks=[0, 1, 2, 3, 4],
    x_as_percent=True,
)
stats_rows.append({"measure": "accuracy", **cmp_accuracy})

# Combined 1x3 figure
fig, axes = plt.subplots(1, 3, figsize=(12.6, 3.4))
plot_cohort_density(
    logrt_by, xlabel="log RT (s)", out_path=OUT_DIR / "_unused.pdf", ax=axes[0], xlim=(-2, 2)
)
plot_cohort_density(
    dprime_by, xlabel="d'", out_path=OUT_DIR / "_unused.pdf", ax=axes[1]
)
plot_cohort_density(
    accuracy_by,
    xlabel="% correct",
    out_path=OUT_DIR / "_unused.pdf",
    ax=axes[2],
    xlim=(0.35, 1.05),
    ylim=(0.0, 5.0),
    yticks=[0, 1, 2, 3, 4],
    x_as_percent=True,
)
fig.tight_layout()
combined_path = OUT_DIR / "cohort_logrt_dprime_density.pdf"
fig.savefig(combined_path, dpi=PUB_DPI, bbox_inches="tight")
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)
print("wrote", combined_path)

stats_df = pd.DataFrame(stats_rows)
stats_path = OUT_DIR / "cohort_comparison_stats.csv"
stats_df.to_csv(stats_path, index=False)
display(stats_df.round(4))
print("wrote", stats_path)


## Notes

- No modeling; log-RT and accuracy densities include all subjects with valid values in the source CSVs.
- log RT uses natural log of positive `gamble_rt` from the learning task only.
- d′ is Day-1 subject-level recognition sensitivity (`dprime`), not `image_dprime` / image `DPRIME`.
- Memory (d′) density / Welch test exclude subjects with d′ < 0 in both cohorts.
- Learning accuracy is the subject mean of trial-wise correctness: `gamble=0` in odd blocks, `gamble=1` in even blocks (0–1 proportion; x-axis shown as %).
- Welch *t*-test (unequal variances) on subject-level scalars; `n.s.` when p >= 0.05.
- Figures save as PDF at `PUB_DPI=300`.
